https://github.com/pytorch/vision/tree/main/references/video_classification

In [1]:
%matplotlib inline

import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE" # tells OpenMP to not complain if it notices that two copies of OpenMP are loaded.

import sys
sys.path.append('../../../')

from computer_vision.misc import download_image_to_numpy

In [12]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import torch
from torchvision.datasets import UCF101
from torchcodec.encoders import VideoEncoder

from IPython.display import Video

def play_video(encoded_bytes):
    return Video(data=encoded_bytes.numpy().tobytes(),
                embed=True, width=640, height=360, mimetype="video/mp4")

In [3]:
# data_dirpath=Path('D:/data/UCF101')
# root=data_dirpath/'UCF-101'
# annotation_path=data_dirpath/'UCF101TrainTestSplits-RecognitionTask'

# # For training, frames_per_clip=16/32 (but 32 requires more memory), step_between_clips=1-5 or 8 needed 
# # to allow overlapping to provide more views of the same action, but smaller step_between_clips could means larger training pool and
# # longer training time
# train_dataset=UCF101(root=root, annotation_path=annotation_path, frames_per_clip=16, step_between_clips=2, train=True) 

# # For evaluation, frames_per_clip=16/32 (but 32 requires more memory), step_between_clips=16/32 for non-overlapping video clips 
# val_dataset=UCF101(root=root, annotation_path=annotation_path, frames_per_clip=16, step_between_clips=16, train=False) 

## See https://www.kaggle.com/code/pevogam/starter-ucf101-with-pytorch

In [4]:
# idx=100
# video, audio, info, video_idx = train_dataset.video_clips.get_clip(idx)
# print(f'{type(video)=}, {type(audio)=}, {type(info)=},{type(video_idx)=}')

In [5]:
file_path = "video_list.txt"
with open(file_path, 'r', encoding='utf-8') as file: video_list = file.read()
video_list=video_list.split('\n')

In [6]:
%load_ext autoreload
%autoreload 2
    
from computer_vision.torch_video.data.deprecated_dataset import VideoClip
from computer_vision.torch_video.data.utils import unfold

In [7]:
import os
import math
import bisect
import warnings

from pathlib import Path
from typing import Any, Callable, Optional, Union, cast, TypeVar

T = TypeVar("T")

import torch
from torchvision.io import _probe_video_from_file, _read_video_from_file, read_video, read_video_timestamps


In [9]:
num_workers=0
frames_per_clip=16
step_between_clips=2
idx=196
metadata_path=r'D:\data\UCF101\deprecated_metadata.pt'
vid_clip=VideoClip(video_paths=video_list, clip_length_in_frames=frames_per_clip,frames_between_clips=step_between_clips,
                   num_workers=num_workers,metadata_path=metadata_path)
video, audio, info, video_idx=vid_clip.get_clip(idx=idx)
print(f"{video.shape=}, {video.dtype=}")
print(f"{audio.shape=}, {audio.dtype=}")
print(f"{info=}")
print(f"{video_idx=}")

video.shape=torch.Size([16, 3, 240, 320]), video.dtype=torch.uint8
audio.shape=torch.Size([2, 18432]), audio.dtype=torch.float32
info={'video_fps': 25.0, 'audio_fps': 44100}
video_idx=2


C:\Users\Sureerat\miniforge3\envs\pytorchvideo\lib\site-packages\torchvision\io\_video_deprecation_warning.py:9: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(
C:\Users\Sureerat\miniforge3\envs\pytorchvideo\lib\site-packages\torchvision\io\video.py:199: UserWarning: The pts_unit 'pts' gives wrong results. Please use pts_unit 'sec'.
  warnings.warn("The pts_unit 'pts' gives wrong results. Please use pts_unit 'sec'.")


In [14]:
encoder=VideoEncoder(frames=video, frame_rate=info['video_fps']) # frame_rate is the frame rate of input video
encoded_frames=encoder.to_tensor(format='mp4')
# play_video(encoded_frames)

In [52]:
from computer_vision.torch_video.data.dataset import VisionDataset
from computer_vision.torch_video.data.utils import find_classes, has_file_allowed_extension, make_dataset

class UCF101(VisionDataset):
    def __init__(self, root:Union[str, Path], annotation_path:str, frames_per_clip:int, step_between_clips:int=1,
                frame_rate:Optional[int]=None,fold:int=1, train:bool=True, transforms:Optional[Callable]=None,
                num_workers:int=1, metadata_path:str=None)->None:
        super().__init__(root, transforms=transforms)

        if not 1<=fold<=3: raise ValueError(f"Fold should be between 1 and 3, but got {fold}")

        extensions=('avi',)
        self.fold=fold
        self.train=train

        self.classes, class_to_idx=find_classes(self.root)
        self.samples=make_dataset(self.root, class_to_idx, extensions, is_valid_file=None)
        video_list=[x[0] for x in self.samples]

        video_clips=VideoClip(video_paths=video_list, clip_length_in_frames=frames_per_clip,frames_between_clips=step_between_clips,
                          num_workers=num_workers,metadata_path=metadata_path)

        # We bookkeep the full version of video clips because we want to be able to return the metadata of full version rather than the
        # subset version of video clips
        self.full_video_clips=video_clips
        self.indices=self._select_fold(video_list, annotation_path, fold, train)
        self.video_clips=video_clips.subset(self.indices)
        self.transforms=transforms

    def _select_fold(self, video_list:list[str], annotation_path:str, fold:int, train:bool)->list[int]:
        """Read txt file listing video files for the specified fold and find the indices of those files in all `video_list`
        Args:
            video_list (list[str]): List of all absolute paths to all video files
            annotation_path (str): Path to directory containing annotation txt file for each fold
            fold (int): Data fold with options of 1, 2 or 3
            train (bool): Whether data is for training or testing
        Returns:
            (list[int]): Indices of selected video from video_list
        """
        name='train' if train else 'test'
        name=f"{name}list{fold:02d}.txt"
        f=os.path.join(annotation_path, name)
        selected_files=set()
        with open(f) as fid:
            data=fid.readlines()
            data=[x.strip().split(" ")[0] for x in data]
            data=[os.path.join(self.root, *x.split("/")) for x in data]
            selected_files.update(data)
        indices=[i for i in range(len(video_list)) if video_list[i] in selected_files]
        return indices

    @property
    def metadata(self)->dict[str, Any]:
        return self.full_video_clips.metadata

    def __len__(self)->int: return self.video_clips.num_clips()

    def __getitem__(self, idx:int)->tuple[torch.Tensor, torch.Tensor, int]:
        video, audio, info, video_idx=self.video_clips.get_clip(idx)
        label=self.samples[self.indices[video_idx]][1]
        info['name']=self.video_clips.video_paths[video_idx]
        if self.transforms is not None: video=self.transforms(video)
        return video, audio, label, info, video_idx
        
data_dirpath=Path('D:/data/UCF101')
root=data_dirpath/'UCF-101'
annotation_path=data_dirpath/'UCF101TrainTestSplits-RecognitionTask'
metadata_path=str(data_dirpath/'deprecated_metadata.pt')
dataset=UCF101(root, annotation_path, frames_per_clip=16, step_between_clips=2, fold=1, train=True, transforms=None,
                num_workers=1, metadata_path=metadata_path)

In [53]:
video, audio, label, info, video_idx=dataset[0]
print(f"{video.shape=}, {video.dtype=}")
print(f"{audio.shape=}, {audio.dtype=}")
print(f"{label=}")
print(f"{info=}")
print(f"{video_idx=}")

video.shape=torch.Size([16, 3, 240, 320]), video.dtype=torch.uint8
audio.shape=torch.Size([2, 18432]), audio.dtype=torch.float32
label=0
info={'video_fps': 25.0, 'audio_fps': 44100, 'name': 'D:\\data\\UCF101\\UCF-101\\ApplyEyeMakeup\\v_ApplyEyeMakeup_g08_c01.avi'}
video_idx=0


In [55]:
idx=0
video, audio, info, video_idx=dataset.video_clips.get_clip(idx)
print(f"{video_idx=}, {dataset.indices[video_idx]=} {dataset.video_clips.video_paths[video_idx]=}")
dataset.samples[dataset.indices[video_idx]]

video_idx=0, dataset.indices[video_idx]=44 dataset.video_clips.video_paths[video_idx]='D:\\data\\UCF101\\UCF-101\\ApplyEyeMakeup\\v_ApplyEyeMakeup_g08_c01.avi'


('D:\\data\\UCF101\\UCF-101\\ApplyEyeMakeup\\v_ApplyEyeMakeup_g08_c01.avi', 0)